In [ ]:
import skimage
import matplotlib.pyplot as plt
import hyperspy.api as hs
import numpy as np
import sys
sys.path.append('..')
import roi_tools

# Buena
s_original = hs.load('../data/images/Jaume LFO/HAADF_Buena.dm3')

left_bound = 20
right_bound = 2048-20
start_pixel = 200
end_pixel = 2048-100

s = s_original.isig[left_bound:right_bound, start_pixel:end_pixel]

In [ ]:
grid_creator = roi_tools.GridCreator()
layout = grid_creator.detect_layout(s)
grid = grid_creator.process(s, layout=layout)
grid = roi_tools.equalize_patch_sizes(grid)
mean_intensities = grid.values(metric='mean_intensity')

fig, ax = plt.subplots(figsize=(10, 10))
plotter = roi_tools.PatchGridPlotter(s, grid, ax=ax)
artist = plotter.add_patch_faces(
    atom_type='Lu',
    values=mean_intensities,
    cmap='viridis',
    label='Mean Intensity',
    alpha=0.7,
)
fig.colorbar(artist, ax=ax, fraction=0.02, label='Mean Intensity')
ax.axis('off')
fig.tight_layout()
plt.show()

In [ ]:
dis = np.array([0, 0, 0, -1, -1, -1, 1, 1, 1, -2, -2, -2, 2, 2, 2])
djs = np.array([-1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1])
djs = djs * 2
offsets = list(zip(dis, djs))


def get_metric(grid, index):
    patch = grid[index]
    vicinal_intensity = 0
    size = len(offsets)
    for di, dj in offsets:
        ni, nj = index[0] + di, index[1] + dj
        if 0 <= ni < grid.shape[0] and 0 <= nj < grid.shape[1]:
            vicinal_intensity += grid[ni, nj].mean_intensity
        else:
            return None
    I_vic = vicinal_intensity / size
    I = patch.mean_intensity
    metric = (I_vic - I) / I_vic if I_vic != 0 else None
    print(
        f"Patch at index {index} has mean intensity {I:.4f}, "
        f"vicinal mean intensity {I_vic:.4f}, and metric {metric:.4f}"
    )
    return metric


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tracked_mult3 = []
tracked_other = []

for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        patch = grid[i, j]
        metric = get_metric(grid, (i, j))

        if grid.atom_types[i, j] == 'Lu' and metric is not None:
            if i % 3 == 2:
                tracked_mult3.append(metric)
            else:
                tracked_other.append(metric)

tracked_mult3 = np.array(tracked_mult3)
tracked_other = np.array(tracked_other)

all_tracked = np.concatenate([tracked_mult3, tracked_other])
bins = np.histogram(all_tracked, bins=50)[1]

plt.figure(figsize=(10, 6))
plt.hist(tracked_mult3, bins=bins, alpha=0.6, color='b', label='Lu down')
plt.hist(tracked_other, bins=bins, alpha=0.6, color='orange', label='Lu up')
plt.title('$(I_{{vic}}-I)/I_{{vic}}$ (Experimental Picture)', fontsize=14)
plt.xlabel('Metric Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(4, 6))

# Added orientation='horizontal' to both calls
plt.hist(tracked_mult3, bins=bins, orientation='horizontal', alpha=0.6, color='b', label='Lu down')
plt.hist(tracked_other, bins=bins, orientation='horizontal', alpha=0.6, color='orange', label='Lu up')

plt.title('$(I_{{vic}}-I)/I_{{vic}}$ (Experimental Picture)', fontsize=14)

# Swapped the labels to match the new orientation
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)

plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hyperspy.api as hs

tracked_mult3 = []
tracked_other = []
grid_creator = roi_tools.GridCreator()
layout_signal = hs.load('../data/simulations/pristine/Iter0_noisy.npy')
layout = grid_creator.detect_layout(layout_signal)

for k in range(30):
    filepath = f'../data/simulations/pristine/Iter{k}_noisy.npy'
    s = hs.load(filepath)

    grid = grid_creator.process(s, layout=layout)
    grid = roi_tools.equalize_patch_sizes(grid)

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            patch = grid[i, j]
            metric = get_metric(grid, (i, j))

            if grid.atom_types[i, j] == 'Lu' and metric is not None:
                if i % 3 == 0:
                    tracked_mult3.append(metric)
                else:
                    tracked_other.append(metric)

tracked_mult3 = np.array(tracked_mult3)
tracked_other = np.array(tracked_other)

all_tracked = np.concatenate([tracked_mult3, tracked_other])
bins = np.histogram(all_tracked, bins=50)[1]

plt.figure(figsize=(10, 6))
plt.hist(tracked_mult3, bins=bins, alpha=0.6, color='b', label='Lu down')
plt.hist(tracked_other, bins=bins, alpha=0.6, color='orange', label='Lu up')
plt.title('$(I_{{vic}}-I)/I_{{vic}}$ (All 30 Simulations)', fontsize=14)
plt.xlabel('Metric Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(4, 6))

# Added orientation='horizontal' to both calls
plt.hist(tracked_mult3, bins=bins, orientation='horizontal', alpha=0.6, color='b', label='Lu down')
plt.hist(tracked_other, bins=bins, orientation='horizontal', alpha=0.6, color='orange', label='Lu up')

plt.title('$(I_{{vic}}-I)/I_{{vic}}$ (All 30 Simulations)', fontsize=14)

# Swapped the labels to match the new orientation
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Metric Value', fontsize=12)

plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
plotter = roi_tools.PatchGridPlotter(s_crop, grid, ax=ax)
mean_intensities = grid.values(metric='mean_intensity')
# artist = plotter.add_patch_faces(atom_type='Lu', values=mean_intensities, cmap='viridis', label='Mean Intensity', alpha=0.7)
# fig.colorbar(artist, ax=ax, fraction=0.05, label='Mean Intensity')
ax.axis('off')
fig.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hyperspy.api as hs
from scipy.spatial import cKDTree

tracked_mult3_voronoi = []
tracked_other_voronoi = []
grid_creator = roi_tools.GridCreator()
layout_signal = hs.load('../data/simulations/pristine/Iter0_noisy.npy')
layout = grid_creator.detect_layout(layout_signal)
fitter = roi_tools.PositionAnalyzer()

for k in range(30):
    filepath = f'../data/simulations/pristine/Iter{k}_noisy.npy'
    s = hs.load(filepath)

    grid = grid_creator.process(s, layout=layout)
    grid = roi_tools.equalize_patch_sizes(grid)
    positions = fitter.fit(grid)

    H, W = s.data.shape
    flat_atoms = []
    atom_indices = []

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            patch = grid[i, j]
            atom_position = positions[i, j]
            if not np.isfinite(atom_position).all():
                continue
            flat_atoms.append(atom_position)
            atom_indices.append((i, j, grid.atom_types[i, j]))

    tree = cKDTree(flat_atoms)
    Y, X = np.mgrid[0:H, 0:W]
    coords = np.column_stack((X.ravel(), Y.ravel()))

    _, nearest_atom_idx = tree.query(coords)
    label_img = nearest_atom_idx.reshape((H, W))

    max_i = grid.shape[0] - 1
    max_j = grid.shape[1] - 1

    for idx, (i, j, atom_type) in enumerate(atom_indices):
        if i == 0 or i == max_i or j == 0 or j == max_j:
            continue

        if atom_type == 'Lu':
            cell_mask = (label_img == idx)
            mean_intensity = np.mean(s.data[cell_mask])

            if i % 3 == 0:
                tracked_mult3_voronoi.append(mean_intensity)
            else:
                tracked_other_voronoi.append(mean_intensity)

tracked_mult3_voronoi = np.array(tracked_mult3_voronoi)
tracked_other_voronoi = np.array(tracked_other_voronoi)

all_tracked = np.concatenate([tracked_mult3_voronoi, tracked_other_voronoi])
bins = np.histogram(all_tracked, bins=50)[1]

plt.figure(figsize=(10, 6))
plt.hist(tracked_mult3_voronoi, bins=bins, alpha=0.6, color='b', label='Lu Atoms (Voronoi, Row % 3 == 0)')
plt.hist(tracked_other_voronoi, bins=bins, alpha=0.6, color='r', label='Lu Atoms (Voronoi, Row % 3 != 0)')
plt.title('Distribution of Mean Intensities (Voronoi Tessellation)', fontsize=14)
plt.xlabel('Mean Intensity', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hyperspy.api as hs

tracked_mult3_circ = []
tracked_other_circ = []

LU_RADIUS_NM = 11
grid_creator = roi_tools.GridCreator()
layout_signal = hs.load('../data/simulations/pristine/Iter0_noisy.npy')
layout = grid_creator.detect_layout(layout_signal)
fitter = roi_tools.PositionAnalyzer()

for k in range(30):
    filepath = f'../data/simulations/pristine/Iter{k}_noisy.npy'
    s = hs.load(filepath)

    grid = grid_creator.process(s, layout=layout)
    grid = roi_tools.equalize_patch_sizes(grid)
    positions = fitter.fit(grid)

    pixel_size = s.axes_manager[0].scale
    radius_px = LU_RADIUS_NM / pixel_size

    if k == 0:
        print(f"Sanity Check: LU_RADIUS_NM ({LU_RADIUS_NM}) / scale ({pixel_size:.4f}) = {radius_px:.2f} pixels.")
        if radius_px < 1.0:
            print("WARNING: Your radius is less than 1 pixel. Consider increasing LU_RADIUS_NM!")

    H, W = s.data.shape
    Y_grid, X_grid = np.mgrid[0:H, 0:W]

    max_i = grid.shape[0] - 1
    max_j = grid.shape[1] - 1

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if i == 0 or i == max_i or j == 0 or j == max_j:
                continue

            patch = grid[i, j]

            atom_position = positions[i, j]
            if grid.atom_types[i, j] == 'Lu' and np.isfinite(atom_position).all():
                x0, y0 = atom_position

                if np.isnan(x0) or np.isnan(y0):
                    continue

                dist_sq = (X_grid - x0)**2 + (Y_grid - y0)**2
                circular_mask = dist_sq <= (radius_px**2)

                if np.sum(circular_mask) == 0:
                    continue

                mean_int = np.mean(s.data[circular_mask])

                if i % 3 == 0:
                    tracked_mult3_circ.append(mean_int)
                else:
                    tracked_other_circ.append(mean_int)

tracked_mult3_circ = np.array(tracked_mult3_circ)
tracked_other_circ = np.array(tracked_other_circ)

tracked_mult3_circ = tracked_mult3_circ[~np.isnan(tracked_mult3_circ)]
tracked_other_circ = tracked_other_circ[~np.isnan(tracked_other_circ)]

all_tracked = np.concatenate([tracked_mult3_circ, tracked_other_circ])
bins = np.histogram(all_tracked, bins=50)[1]

plt.figure(figsize=(10, 6))
plt.hist(tracked_mult3_circ, bins=bins, alpha=0.6, color='b', label='Lu Atoms (Circular, Row % 3 == 0)')
plt.hist(tracked_other_circ, bins=bins, alpha=0.6, color='r', label='Lu Atoms (Circular, Row % 3 != 0)')
plt.title(f'Distribution of Mean Intensities (Circular Mask, r={LU_RADIUS_NM} pixels)', fontsize=14)
plt.xlabel('Mean Intensity', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()
